# Speech Psychology — Sentence Transformer Model
**روش:** Sentence Embeddings (all-MiniLM-L6-v2) + Ridge Regression

برخلاف TF-IDF که فقط کلمات رو می‌بیند، این مدل **معنای جمله** رو درک می‌کند.

## 1. Install & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
import pickle

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

## 2. Load Data

In [ ]:
train = pd.read_csv('comments/train.csv')
test  = pd.read_csv('comments/test.csv')

TARGET_COLS = [
    'sense', 'honor', 'curse', 'despise', 'situation',
    'antihuman', 'roughness', 'slaughter', 'strike_support', 'depression_rate'
]

X_train_text = train['text'].fillna('').astype(str).tolist()
y_train      = train[TARGET_COLS].values
X_test_text  = test['text'].fillna('').astype(str).tolist()

print(f'Train: {len(X_train_text)} samples')
print(f'Test:  {len(X_test_text)} samples')

## 3. Load Sentence Transformer Model

`all-MiniLM-L6-v2` یه مدل سبک (80MB) است که:
- هر متن رو به یه بردار **384 بُعدی** تبدیل می‌کند
- معنای جمله رو capture می‌کند (نه فقط کلمات)
- روی CPU هم در زمان معقول اجرا می‌شود
- از متون چندزبانه هم نسبتاً خوب پشتیبانی می‌کند

In [ ]:
print('Loading model... (اولین بار دانلود می‌شود ~80MB)')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded!')

## 4. Generate Embeddings

هر متن به یه بردار عددی 384 بُعدی تبدیل می‌شود.

⚠️ روی CPU این مرحله حدود **10-20 دقیقه** طول می‌کشد — صبور باشید!
نتیجه رو ذخیره می‌کنیم تا دفعات بعد نیازی به محاسبه مجدد نباشد.

In [ ]:
import os

# اگر embedding ها قبلاً ذخیره شده، لود کن — وگرنه محاسبه کن
if os.path.exists('train_embeddings.npy') and os.path.exists('test_embeddings.npy'):
    print('Loading saved embeddings...')
    X_train_emb = np.load('train_embeddings.npy')
    X_test_emb  = np.load('test_embeddings.npy')
    print('Loaded!')
else:
    print('Encoding train set...')
    t0 = time.time()
    X_train_emb = embedder.encode(
        X_train_text,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    print(f'Train encoded in {(time.time()-t0)/60:.1f} min')

    print('Encoding test set...')
    t0 = time.time()
    X_test_emb = embedder.encode(
        X_test_text,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    print(f'Test encoded in {(time.time()-t0)/60:.1f} min')

    # ذخیره برای دفعات بعد
    np.save('train_embeddings.npy', X_train_emb)
    np.save('test_embeddings.npy',  X_test_emb)
    print('Embeddings saved!')

print(f'Train embeddings shape: {X_train_emb.shape}')
print(f'Test embeddings shape:  {X_test_emb.shape}')

## 5. Normalize Embeddings

StandardScaler هر feature رو به میانگین ۰ و انحراف معیار ۱ تبدیل می‌کند.
این کار باعث می‌شود Ridge بهتر همگرا شود.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_emb)
X_test_scaled  = scaler.transform(X_test_emb)

print(f'Scaled — mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}')

## 6. Train Ridge Regression

روی embedding های ۳۸۴ بُعدی یه Ridge Regression می‌ذاریم.
alpha رو با چند مقدار امتحان می‌کنیم.

In [ ]:
# پیدا کردن بهترین alpha با CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print('Searching best alpha...')
alpha_results = {}

for alpha in [0.1, 1.0, 10.0, 100.0]:
    rmse_list = []
    for col_idx, col in enumerate(TARGET_COLS):
        scores = cross_val_score(
            Ridge(alpha=alpha),
            X_train_scaled,
            y_train[:, col_idx],
            cv=kf,
            scoring='neg_root_mean_squared_error'
        )
        rmse_list.append(-scores.mean())
    mcrmse = np.mean(rmse_list)
    alpha_results[alpha] = mcrmse
    print(f'  alpha={alpha:6.1f} → MCRMSE = {mcrmse:.4f}')

best_alpha = min(alpha_results, key=alpha_results.get)
print(f'\nBest alpha: {best_alpha} (MCRMSE={alpha_results[best_alpha]:.4f})')

In [ ]:
# آموزش مدل نهایی با بهترین alpha
model = MultiOutputRegressor(
    Ridge(alpha=best_alpha),
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)
print('Model trained!')

## 7. Evaluation — Per Column RMSE

In [ ]:
rmse_scores = {}

for col_idx, col in enumerate(TARGET_COLS):
    scores = cross_val_score(
        Ridge(alpha=best_alpha),
        X_train_scaled,
        y_train[:, col_idx],
        cv=kf,
        scoring='neg_root_mean_squared_error'
    )
    rmse_scores[col] = -scores.mean()

results_df = pd.DataFrame({
    'Column': list(rmse_scores.keys()),
    'RMSE':   list(rmse_scores.values())
}).sort_values('RMSE', ascending=False)

print(results_df.to_string(index=False))

mcrmse = results_df['RMSE'].mean()
score  = (1.5 - mcrmse) * (100/150) * 150
print(f'\nMCRMSE: {mcrmse:.4f}')
print(f'Estimated Score: {score:.2f} / 150')

In [ ]:
# مقایسه baseline vs این مدل
baseline_mcrmse = 0.7757  # از مرحله قبل

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(results_df['Column'], results_df['RMSE'], color='steelblue', edgecolor='white')
ax.axvline(x=1.5,              color='red',    linestyle='--', label='Reject threshold (1.5)')
ax.axvline(x=baseline_mcrmse,  color='orange', linestyle='--', label=f'Baseline MCRMSE ({baseline_mcrmse})')
ax.axvline(x=mcrmse,           color='green',  linestyle='--', label=f'This model MCRMSE ({mcrmse:.3f})')
ax.set_xlabel('RMSE')
ax.set_title('RMSE per Column — Sentence Transformer + Ridge')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Predict & Save Output

In [ ]:
# پیش‌بینی روی test
preds = model.predict(X_test_scaled)

# مقادیر رو در بازه [0, 4] نگه می‌داریم
preds = np.clip(preds, 0, 4)

output = pd.DataFrame(preds, columns=TARGET_COLS).round(3)

print(f'Output shape: {output.shape}')  # باید (1754, 10) باشد
output.head()

In [ ]:
output.to_csv('bert_output.csv', index=False)
print(f'bert_output.csv saved — {len(output)} rows, {len(output.columns)} columns')

## 9. مقایسه Baseline vs این مدل

| | Baseline | این مدل |
|---|---|---|
| روش | TF-IDF + Ridge | Sentence Embedding + Ridge |
| درک معنا | ❌ | ✅ |
| چندزبانه | ❌ | نسبی ✅ |
| MCRMSE | 0.776 | ... |

**نقاط ضعف باقی‌مانده:**
- مدل هنوز fine-tune نشده روی این task
- متون غیرانگلیسی هنوز ضعیف‌تر handle می‌شوند

**مرحله بعد (اختیاری):** Fine-tune کردن کامل مدل transformer روی این dataset